# Inteligencia Artificial — Tarea 4
**Puntos 1 y 2:** convolución a mano, datos y MLP.     Integrantes: Laura Ruidiaz y Angelo Maiale

Estructura del proyecto:
- `convolucion.py`: convolución, ReLU y max pooling implementados a mano (punto 1).
- `datos.py`: semilla, carga de FashionMNIST, separación train/val/test y gráficas (punto 2).
- `modelos.py`: clase `MLP(nn.Module)`.
- `entrenamiento.py`: ciclo de entrenamiento, evaluación y curvas.

In [ ]:
import inspect
import torch
import torch.nn as nn

from datos import (SEMILLA, NOMBRES_CLASES, fijar_semilla, obtener_dispositivo, cargar_datasets,
                   crear_dataloaders, mostrar_ejemplos, proporcion_por_clase, obtener_etiquetas)
from convolucion import (convolucion_manual, relu_manual, maxpool_manual, explicar_convolucion,
                         explicar_maxpool, parametros_conv, parametros_densa)
from modelos import MLP
from entrenamiento import entrenar, graficar_curvas, contar_parametros

fijar_semilla(SEMILLA)
dispositivo = obtener_dispositivo()
print("Semilla:", SEMILLA, "| Dispositivo:", dispositivo)

---
## Punto 1. Convolución a mano

$$
X=\begin{pmatrix}1&0&2&1&0\\0&1&1&0&2\\2&3&0&1&1\\1&1&0&2&1\\0&1&2&1&0\end{pmatrix},\qquad
K=\begin{pmatrix}1&1&1\\0&0&0\\-1&-1&-1\end{pmatrix}
$$

Con stride 1 y sin padding, la salida mide $(5-3)/1+1 = 3$ en cada dimensión, es decir, **3 × 3**.
Cada elemento es la suma de los productos elemento a elemento entre el filtro y la ventana $3\times3$ de la imagen.
(PyTorch no voltea el filtro: calcula una correlación cruzada.)

Como $K$ tiene unos arriba, ceros en el medio y menos unos abajo, cada salida es
*(suma de la fila superior de la ventana) − (suma de la fila inferior)*: un detector de bordes horizontales.

In [ ]:
X = torch.tensor([[1, 0, 2, 1, 0],
                  [0, 1, 1, 0, 2],
                  [2, 3, 0, 1, 1],
                  [1, 1, 0, 2, 1],
                  [0, 1, 2, 1, 0]], dtype=torch.float32)
K = torch.tensor([[ 1,  1,  1],
                  [ 0,  0,  0],
                  [-1, -1, -1]], dtype=torch.float32)

print("Cálculo paso a paso de la convolución (stride 1, sin padding, sin sesgo):")
explicar_convolucion(X, K, stride=1)

### Resultado de la convolución (a mano)

| | col 0 | col 1 | col 2 |
|---|---|---|---|
| **fila 0** | $3-5=-2$ | $3-4=-1$ | $3-2=1$ |
| **fila 1** | $2-2=0$ | $2-3=-1$ | $3-3=0$ |
| **fila 2** | $5-3=2$ | $4-4=0$ | $2-3=-1$ |

$$
\text{Conv}(X,K)=\begin{pmatrix}-2&-1&1\\0&-1&0\\2&0&-1\end{pmatrix}
$$

### ReLU

Se reemplazan los negativos por 0:

$$
\text{ReLU}=\begin{pmatrix}0&0&1\\0&0&0\\2&0&0\end{pmatrix}
$$

### Max pooling 2 × 2, stride 1

La entrada es $3\times3$, así que la salida mide $(3-2)/1+1=2$ por lado: **2 × 2**.

$$
\text{MaxPool}=\begin{pmatrix}\max(0,0,0,0)&\max(0,1,0,0)\\\max(0,0,2,0)&\max(0,0,0,0)\end{pmatrix}=\begin{pmatrix}0&1\\2&0\end{pmatrix}
$$

In [ ]:
# Mismo cálculo con las funciones hechas a mano
conv_manual = convolucion_manual(X, K, stride=1, padding=0)
relu_resultado = relu_manual(conv_manual)
print("Convolución:\n", conv_manual)
print("\nReLU:\n", relu_resultado)
print("\nMax pooling 2x2, stride 1 (paso a paso):")
explicar_maxpool(relu_resultado, tam=2, stride=1)
pool_manual = maxpool_manual(relu_resultado, tam=2, stride=1)
print("\nResultado:\n", pool_manual)

# Resultados obtenidos a mano en las celdas de arriba
esperado_conv = torch.tensor([[-2., -1., 1.], [0., -1., 0.], [2., 0., -1.]])
esperado_pool = torch.tensor([[0., 1.], [2., 0.]])
assert torch.equal(conv_manual, esperado_conv)
assert torch.equal(pool_manual, esperado_pool)
print("\nLas funciones manuales coinciden con el cálculo hecho a mano.")

### Verificación con `nn.Conv2d` y `nn.MaxPool2d` (pesos asignados a mano)

In [ ]:
conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, stride=1, padding=0, bias=False)
with torch.no_grad():
    conv.weight.copy_(K.view(1, 1, 3, 3))   # forma (c_out, c_in, kh, kw)

pool = nn.MaxPool2d(kernel_size=2, stride=1)
relu = nn.ReLU()

with torch.no_grad():
    x = X.view(1, 1, 5, 5)                  # (batch, canales, alto, ancho)
    salida_conv = conv(x)
    salida_pool = pool(relu(salida_conv))

print("nn.Conv2d:\n", salida_conv.squeeze())
print("\nnn.MaxPool2d tras ReLU:\n", salida_pool.squeeze())

print("\n¿Conv coincide con el cálculo a mano? ", torch.allclose(salida_conv.squeeze(), esperado_conv))
print("¿Pool coincide con el cálculo a mano? ", torch.allclose(salida_pool.squeeze(), esperado_pool))

### Parámetros: capa convolucional vs. capa densa

**`nn.Conv2d(3, 16, kernel_size=3)`**: cada uno de los 16 filtros tiene $3\times3\times3=27$ pesos más un sesgo:
$$16\cdot(3\cdot3\cdot3)+16 = 432+16 = \mathbf{448}$$

**Capa densa** de una imagen $32\times32\times3$ (3 072 entradas) a una salida $30\times30\times16$ (14 400 neuronas): cada salida se conecta con todas las entradas:
$$3072\cdot14400+14400=44\,236\,800+14\,400=\mathbf{44\,251\,200}$$

La convolución usa **conexiones locales** (cada salida ve solo una ventana $3\times3$) y **pesos compartidos** (el mismo filtro se desplaza por toda la imagen), por eso necesita ~100 000 veces menos parámetros.

In [ ]:
conv_capa = nn.Conv2d(3, 16, kernel_size=3)
densa = nn.Linear(32 * 32 * 3, 30 * 30 * 16)

p_conv = sum(p.numel() for p in conv_capa.parameters())
p_densa = sum(p.numel() for p in densa.parameters())

print("Conv2d(3, 16, 3):")
for nombre, p in conv_capa.named_parameters():
    print(f"  {nombre}: {tuple(p.shape)} -> {p.numel()}")
print(f"  total (p.numel()): {p_conv} | fórmula: {parametros_conv(3, 16, 3)}")

print("\nCapa densa 3072 -> 14400:")
for nombre, p in densa.named_parameters():
    print(f"  {nombre}: {tuple(p.shape)} -> {p.numel()}")
print(f"  total (p.numel()): {p_densa:,} | fórmula: {parametros_densa(3072, 14400):,}")

print(f"\nLa capa densa tiene {p_densa / p_conv:,.0f} veces más parámetros.")
del densa

---
## Punto 2. Datos y MLP

**Dataset:** FashionMNIST (`torchvision.datasets`): 10 clases de prendas, imágenes de $28\times28$ en escala de grises.
Tiene 60 000 imágenes de entrenamiento, más de 20 000, así que se toma una **muestra aleatoria de 20 000** (con semilla fija).
Esa muestra se divide en 80 % entrenamiento (16 000) y 20 % validación (4 000). Como prueba se usa el test oficial (10 000 imágenes).
La separación se guarda en `splits.pt` y se reutiliza en todos los modelos de la tarea.

In [ ]:
train_ds, val_ds, test_ds = cargar_datasets(root="data", n_entrenamiento=20000, frac_val=0.2)
print(f"Entrenamiento: {len(train_ds)} | Validación: {len(val_ds)} | Prueba: {len(test_ds)}")
print("Clases:", NOMBRES_CLASES)

### Imágenes de cada clase

In [ ]:
mostrar_ejemplos(train_ds, n_por_clase=6)

### Proporción de ejemplos por clase

In [ ]:
proporcion_por_clase(train_ds, val_ds, test_ds)

Las clases están prácticamente balanceadas (alrededor de 10 % cada una en los tres conjuntos), así que la exactitud es una métrica razonable y un clasificador aleatorio acertaría ~10 %.

In [ ]:
# Mismos dataloaders para todos los modelos de la tarea
BATCH_SIZE = 128
train_loader, val_loader, test_loader = crear_dataloaders(train_ds, val_ds, test_ds, batch_size=BATCH_SIZE)

x, y = next(iter(train_loader))
print("Forma de un mini-batch:", tuple(x.shape), tuple(y.shape))
print("Mini-batches por época (pasos):", len(train_loader))

### Modelo: `MLP(nn.Module)`

In [ ]:
print(inspect.getsource(MLP))

In [ ]:
fijar_semilla(SEMILLA)
mlp = MLP()
print(mlp)
print("\nForma de la salida:", tuple(mlp(x).shape))
print("Parámetros del MLP:", f"{contar_parametros(mlp):,}")
for nombre, p in mlp.named_parameters():
    print(f"  {nombre:<14}{str(tuple(p.shape)):<14}{p.numel():>9,}")

### Entrenamiento

In [ ]:
EPOCAS = 20
LR = 1e-3
print(f"Pasos totales: {len(train_loader)} por época x {EPOCAS} épocas = {len(train_loader) * EPOCAS}\n")

fijar_semilla(SEMILLA)
mlp = MLP()
hist_mlp = entrenar(mlp, train_loader, val_loader, epocas=EPOCAS, lr=LR, dispositivo=dispositivo)

### Curvas de pérdida y exactitud

In [ ]:
graficar_curvas(hist_mlp, titulo="MLP: entrenamiento vs. validación")

In [ ]:
mejor = max(range(EPOCAS), key=lambda i: hist_mlp["val_acc"][i])
print(f"Última época  -> exactitud train {hist_mlp['train_acc'][-1]:.4f} | val {hist_mlp['val_acc'][-1]:.4f}")
print(f"Mejor val_acc -> época {mejor + 1}: {hist_mlp['val_acc'][mejor]:.4f}")

**Observaciones** (completar con las curvas obtenidas al ejecutar):
- ¿Qué pasa con la pérdida de entrenamiento y la de validación? Si la de validación deja de bajar o sube mientras la de entrenamiento sigue bajando, hay sobreajuste.
- ¿Cuál es la brecha entre la exactitud de entrenamiento y la de validación?

> La exactitud de **prueba** no se evalúa aquí: el enunciado pide reportarla una sola vez, al final de la tarea (punto 3).

In [ ]:
torch.save(mlp.state_dict(), "mlp.pt")
print("Pesos del MLP guardados en mlp.pt")